## **Generate and visualise a state sequence**
Author: patrick.mccarthy@dpag.ox.ac.uk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from cxval.tasks import ValueMatrix, StimulusSequence, StateSequence
from cxval.vis import STYLE
import matplotlib.pyplot as plt
import numpy as np

plt.style.use(STYLE)

Generate a value matrix, stimulus sequence, and state sequence

In [ ]:
num_stimuli = 4
num_contexts = 3
stimuli = [f"s{i}" for i in range(num_stimuli)]
contexts = [f"c{j}" for j in range(num_contexts)]

trials_per_phase = 4
phases_per_context = 2
stim_timesteps = 5
reward_timesteps = 3
iti_timesteps = (2, 5)

In [ ]:
val_matrix = ValueMatrix(seed=0, contexts=contexts, stimuli=stimuli,
                         delta_context=0.1, base_lower=0.2, base_upper=0.8)
val_matrix.generate_base_values(seed=0)
mtx = val_matrix.generate_value_matrix(seed=0)

stim_seq = StimulusSequence(value_matrix=mtx, trials_per_phase=trials_per_phase,
                             phases_per_context=phases_per_context, context_order='sequential')
stim_seq.generate(seed=0)

state_seq = StateSequence(stimulus_sequence=stim_seq, value_matrix=mtx,
                          stim_timesteps=stim_timesteps, reward_timesteps=reward_timesteps,
                          iti_timesteps=iti_timesteps)
states, rewards = state_seq.generate(seed=0)

print(f"States shape: {states.shape}")
print(f"Total trials: {len(stim_seq.trial_contexts)}")

Visualise state sequence. An ITI indicator row is appended to the bottom of the heatmap so blank periods are explicit.

In [ ]:
from matplotlib.lines import Line2D

trial_lengths = state_seq.iti_durations + stim_timesteps + reward_timesteps
trial_boundaries = np.concatenate([[0], np.cumsum(trial_lengths)])

# build ITI indicator row (1 during ITI, 0 otherwise)
iti_mask = np.zeros(states.shape[0])
t = 0
for iti_len, tlen in zip(state_seq.iti_durations, trial_lengths):
    iti_mask[t:t + iti_len] = 1.0
    t += tlen

vis_states = np.vstack([states.T, iti_mask])

# trial indices at phase and context boundaries
trials_per_context = trials_per_phase * phases_per_context
n_trials = len(stim_seq.trial_contexts)
phase_ts = trial_boundaries[np.arange(0, n_trials + 1, trials_per_phase)]
context_ts = trial_boundaries[np.arange(0, n_trials + 1, trials_per_context)]
context_ts_set = set(context_ts)

ylabels = ([f"ctx {c}" for c in contexts]
           + [f"stim {s}" for s in stimuli]
           + ["reward", "ITI"])

fig, ax = plt.subplots(figsize=(18, 4))
im = ax.imshow(vis_states, aspect="auto", cmap="hot", vmin=0, vmax=1, interpolation="nearest")

for t in trial_boundaries[1:-1]:
    ax.axvline(t - 0.5, color="gray", linewidth=0.5, alpha=0.4)

for t in phase_ts[1:-1]:
    if t not in context_ts_set:
        ax.axvline(t - 0.5, color="steelblue", linewidth=1.5, linestyle="--", alpha=0.8)

for t in context_ts[1:-1]:
    ax.axvline(t - 0.5, color="white", linewidth=2.5)

ax.set_yticks(np.arange(len(ylabels)))
ax.set_yticklabels(ylabels)
ax.set_xlabel("Timestep")
ax.set_title("State Sequence")
plt.colorbar(im, ax=ax, label="Activation")

legend_elements = [
    Line2D([0], [0], color="gray",     linewidth=0.5,               label="Trial boundary"),
    Line2D([0], [0], color="steelblue", linewidth=1.5, linestyle="--", label="Phase boundary"),
    Line2D([0], [0], color="white",    linewidth=2.5,               label="Context boundary"),
]
leg = ax.legend(handles=legend_elements, loc="upper right", framealpha=0.6,
                facecolor="dimgray", labelcolor="white")

plt.tight_layout()
plt.show()